# Hotel Review Dataset Preprocessing for Chatbot (RAG)

This notebook prepares hotel review data for use in a Retrieval‑Augmented Generation (RAG) chatbot.  
We will:
1. Load the dataset
2. Clean and normalize text
3. Handle missing values
4. Engineer metadata features
5. Consolidate text for embeddings
6. Save processed data for ChromaDB/FAISS indexing


In [1]:
# Import Libraries
import pandas as pd
import re  # Regular Expressions (Regex)
import ast # Abstract Syntax Trees
# Allow unlimited column text width
pd.set_option('display.max_colwidth', None)

#The ast module allows Python to process and parse Python source code as a tree structure of objects.

df = pd.read_csv("C:/agentic-ai/ML_project/Dataset/Hotel_Reviews.csv")

df.head(1)

,Hotel_Address,Additional_Number_of_Scoring,Review_Date,Average_Score,Hotel_Name,Reviewer_Nationality,Negative_Review,Review_Total_Negative_Word_Counts,Total_Number_of_Reviews,Positive_Review,Review_Total_Positive_Word_Counts,Total_Number_of_Reviews_Reviewer_Has_Given,Reviewer_Score,Tags,days_since_review,lat,lng
0,s Gravesandestraat 55 Oost 1092 AA Amsterdam Netherlands,194,8/3/2017,7.7,Hotel Arena,Russia,I am so angry that i made this post available via all possible sites i use when planing my trips so no one will make the mistake of booking this place I made my booking via booking com We stayed for 6 nights in this hotel from 11 to 17 July Upon arrival we were placed in a small room on the 2nd floor of the hotel It turned out that this was not the room we booked I had specially reserved the 2 level duplex room so that we would have a big windows and high ceilings The room itself was ok if you don t mind the broken window that can not be closed hello rain and a mini fridge that contained some sort of a bio weapon at least i guessed so by the smell of it I intimately asked to change the room and after explaining 2 times that i booked a duplex btw it costs the same as a simple double but got way more volume due to the high ceiling was offered a room but only the next day SO i had to check out the next day before 11 o clock in order to get the room i waned to Not the best way to begin your holiday So we had to wait till 13 00 in order to check in my new room what a wonderful waist of my time The room 023 i got was just as i wanted to peaceful internal garden view big window We were tired from waiting the room so we placed our belongings and rushed to the city In the evening it turned out that there was a constant noise in the room i guess it was made by vibrating vent tubes or something it was constant and annoying as hell AND it did not stop even at 2 am making it hard to fall asleep for me and my wife I have an audio recording that i can not attach here but if you want i can send it via e mail The next day the technician came but was not able to determine the cause of the disturbing sound so i was offered to change the room once again the hotel was fully booked and they had only 1 room left the one that was smaller but seems newer,397,1403,Only the park outside of the hotel was beautiful,11,7,2.9,"[' Leisure trip ', ' Couple ', ' Duplex Double Room ', ' Stayed 6 nights ']",0 days,52.360576,4.915968


In [2]:
# clean text fields
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    
    # Replace placeholder values with empty string
    if text.strip() in ["No Negative", "No Positive"]:
        return ""
    
    # Remove HTML tags
    text = re.sub(r"<.*?>", " ", text)
    # Remove special characters
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Apply cleaning
df["Positive_Review"] = df["Positive_Review"].apply(clean_text)
df["Negative_Review"] = df["Negative_Review"].apply(clean_text)

# Drop blank or null review rows
df = df.dropna(subset=["Positive_Review", "Negative_Review"])   # drop nulls
df = df[(df["Positive_Review"].str.strip() != "") | (df["Negative_Review"].str.strip() != "")]  # drop blanks

# Preview cleaned dataset
df[["Hotel_Name", "Positive_Review", "Negative_Review"]].head(2)


,Hotel_Name,Positive_Review,Negative_Review
0,Hotel Arena,only the park outside of the hotel was beautiful,i am so angry that i made this post available via all possible sites i use when planing my trips so no one will make the mistake of booking this place i made my booking via booking com we stayed for 6 nights in this hotel from 11 to 17 july upon arrival we were placed in a small room on the 2nd floor of the hotel it turned out that this was not the room we booked i had specially reserved the 2 level duplex room so that we would have a big windows and high ceilings the room itself was ok if you don t mind the broken window that can not be closed hello rain and a mini fridge that contained some sort of a bio weapon at least i guessed so by the smell of it i intimately asked to change the room and after explaining 2 times that i booked a duplex btw it costs the same as a simple double but got way more volume due to the high ceiling was offered a room but only the next day so i had to check out the next day before 11 o clock in order to get the room i waned to not the best way to begin your holiday so we had to wait till 13 00 in order to check in my new room what a wonderful waist of my time the room 023 i got was just as i wanted to peaceful internal garden view big window we were tired from waiting the room so we placed our belongings and rushed to the city in the evening it turned out that there was a constant noise in the room i guess it was made by vibrating vent tubes or something it was constant and annoying as hell and it did not stop even at 2 am making it hard to fall asleep for me and my wife i have an audio recording that i can not attach here but if you want i can send it via e mail the next day the technician came but was not able to determine the cause of the disturbing sound so i was offered to change the room once again the hotel was fully booked and they had only 1 room left the one that was smaller but seems newer
1,Hotel Arena,no real complaints the hotel was great great location surroundings rooms amenities and service two recommendations however firstly the staff upon check in are very confusing regarding deposit payments and the staff offer you upon checkout to refund your original payment and you can make a new one bit confusing secondly the on site restaurant is a bit lacking very well thought out and excellent quality food for anyone of a vegetarian or vegan background but even a wrap or toasted sandwich option would be great aside from those minor minor things fantastic spot and will be back when i return to amsterdam,no negative


In [3]:
# Handle missing values
df["Reviewer_Score"] = df["Reviewer_Score"].fillna(df["Reviewer_Score"].mean())
df["Average_Score"] = df["Average_Score"].fillna(df["Average_Score"].mean())
df["Tags"] = df["Tags"].fillna("[]")

# Convert tags from string to list
df["Tags"] = df["Tags"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) and x.startswith("[") else [x])


In [4]:
df[["Tags"]].head(1)

,Tags
0,"[ Leisure trip , Couple , Duplex Double Room , Stayed 6 nights ]"


In [5]:
# Feature Engineering
df["Review_Date"] = pd.to_datetime(df["Review_Date"], errors="coerce")
df["days_since_review"] = pd.to_numeric(df["days_since_review"], errors="coerce")

# Known non-room-type tag categories (derived from inspecting tag frequencies)
GUEST_TYPE_TAGS = {
    "couple", "solo traveler", "group",
    "family with young children", "family with older children",
    "travelers with friends",
}
# \b word boundaries matter here: "trip" as a plain substring also matches
# inside "Triple Room" ("tri-p-le"), which would misclassify room tags.
TRIP_RE = re.compile(r"\btrip\b")


def extract_trip_type(tags):
    """Returns a single clean string, e.g. 'Leisure trip' or 'Business trip'."""
    for t in tags:
        if TRIP_RE.search(t.lower()):
            return t
    return "Unknown"


def extract_guest_type(tags):
    """Returns the guest-type tag, e.g. 'Couple', 'Solo traveler', 'Group'."""
    for t in tags:
        if t.lower() in GUEST_TYPE_TAGS:
            return t
    return "Unknown"


def extract_is_couple(tags):
    return any(t.lower() == "couple" for t in tags)


def extract_nights_stayed(tags):
    for t in tags:
        m = re.search(r"stayed\s+(\d+)\s+night", t.lower())
        if m:
            return int(m.group(1))
    return None


def extract_room_type(tags):
    """Whatever tag is left over after trip type, guest type, nights, mobile-submission,
    and 'N rooms' tags are excluded is treated as the room type."""
    exclude_exact = GUEST_TYPE_TAGS | {"submitted from a mobile device"}
    for t in tags:
        low = t.lower()
        if TRIP_RE.search(low):
            continue
        if low in exclude_exact:
            continue
        if low.startswith("stayed"):
            continue
        if re.match(r"^\d+ rooms?$", low):
            continue
        return t
    return "Unknown"


df["Trip_Type"] = df["Tags"].apply(extract_trip_type)
df["Guest_Type"] = df["Tags"].apply(extract_guest_type)
df["Is_Couple"] = df["Tags"].apply(extract_is_couple)
df["Nights_Stayed"] = df["Tags"].apply(extract_nights_stayed)
df["Room_Type"] = df["Tags"].apply(extract_room_type)


In [6]:
df[["Trip_Type", "Guest_Type", "Is_Couple", "Room_Type", "Nights_Stayed"]].head(5)


,Trip_Type,Guest_Type,Is_Couple,Room_Type,Nights_Stayed
0,Leisure trip,Unknown,False,Couple,6.0
1,Leisure trip,Unknown,False,Couple,4.0
2,Leisure trip,Unknown,False,Family with young children,3.0
3,Leisure trip,Unknown,False,Solo traveler,3.0
4,Leisure trip,Unknown,False,Couple,2.0


In [7]:
# Consolidate text for embeddings
df["combined_text"] = df.apply(
    lambda row: f"Hotel: {row['Hotel_Name']} in {row['Hotel_Address']}. "
                f"Positive review: {row['Positive_Review']}. "
                f"Negative review: {row['Negative_Review']}. "
                f"Tags: {', '.join(row['Tags'])}. "
                f"Average Score: {row['Average_Score']}/10. "
                f"Reviewer Score: {row['Reviewer_Score']}/10.",
    axis=1
)

# Preview
df["combined_text"].head()



0    Hotel: Hotel Arena in  s Gravesandestraat 55 Oost 1092 AA Amsterdam Netherlands. Positive review: only the park outside of the hotel was beautiful. Negative review: i am so angry that i made this post available via all possible sites i use when planing my trips so no one will make the mistake of booking this place i made my booking via booking com we stayed for 6 nights in this hotel from 11 to 17 july upon arrival we were placed in a small room on the 2nd floor of the hotel it turned out that this was not the room we booked i had specially reserved the 2 level duplex room so that we would have a big windows and high ceilings the room itself was ok if you don t mind the broken window that can not be closed hello rain and a mini fridge that contained some sort of a bio weapon at least i guessed so by the smell of it i intimately asked to change the room and after explaining 2 times that i booked a duplex btw it costs the same as a simple double but got way more volume due to the hi

In [8]:
# Save processed dataset
# Absolute path -- must match CSV_PATH in chatbot.py exactly, so the chatbot
# always reads the file this notebook actually produced.
OUTPUT_PATH = "C:/agentic-ai/ML_project/Dataset/processed_hotel_reviews.csv"
df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(df)} rows to {OUTPUT_PATH}")
print("Columns:", df.columns.tolist())


Saved 515679 rows to C:/agentic-ai/ML_project/Dataset/processed_hotel_reviews.csv
Columns: ['Hotel_Address', 'Additional_Number_of_Scoring', 'Review_Date', 'Average_Score', 'Hotel_Name', 'Reviewer_Nationality', 'Negative_Review', 'Review_Total_Negative_Word_Counts', 'Total_Number_of_Reviews', 'Positive_Review', 'Review_Total_Positive_Word_Counts', 'Total_Number_of_Reviews_Reviewer_Has_Given', 'Reviewer_Score', 'Tags', 'days_since_review', 'lat', 'lng', 'Trip_Type', 'Guest_Type', 'Is_Couple', 'Nights_Stayed', 'Room_Type', 'combined_text']
